# Outer Loop Evaluation With MLflow

This notebook demonstrates how to setup MLflow's outer loop evaluation using its functionalities like in-built scorers, LLM-judge and automatic evaluation features to evaluate an agent's performance in production. In order to run this notebook, you will need to have a deployed agent application running on OpenShift AI. The `2_deploy` folder contains instructions which you can follow for deploying your own agent application.

## Prerequisites

- An OpenShift AI cluster
- An agent application running and deployed on OpenShift AI. You can follow the steps in the `2_deploy` folder notebook [here](../2_deploy/deploy.ipynb) for deploying your own agent application.
- Set up a virtual environment
  - Install [uv](https://docs.astral.sh/uv/getting-started/installation/)
  - Create a new virtual environment - `uv venv`
  - Add ipykernel to your venv - `uv pip install ipykernel ipywidgets`
- Have access to an OpenAI-compatible LLM endpoint (e.g. OpenAI, vLLM, llama.cpp, Ollama, etc.)
  - Set the environment variables for your endpoint using the cell below
  - These will be used in a later example with OpenAI Agents SDK

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file (searches current and parent directories)
load_dotenv(find_dotenv())

# Check that required vars are set, raise an error if not
required_vars = ["AGENT_URL", "MLFLOW_TRACKING_URI", "MLFLOW_EXPERIMENT_NAME", "MLFLOW_TRACKING_AUTH", "OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL_NAME"]
if any(not os.getenv(var) for var in required_vars):
    raise ValueError("One or more required environment variables are not set, please set them above.")

## Install Requirements

In [ ]:
# If using RHOAI and mlflow < 3.10 install the Red Hat build of MLflow
![[ "$MLFLOW_TRACKING_AUTH" == "kubernetes" ]] && [[ $(echo -e "$(uv run mlflow --version | awk '{print $NF}')\n3.10.0" | sort -V | head -1) != "3.10.0" ]] && \
    uv pip install "git+https://github.com/opendatahub-io/mlflow@master"

## Start the Local MLflow Server (Optional)

If you are using the local MLflow configuration, open a terminal in the parent directory and run the following command to start a local MLflow server

> ```sh
> uv run mlflow server --port 5001
> ```

Your server will be available on [http://localhost:5001](http://localhost:5001). Please open your browser and go to this URL to confirm the server is running.

## Agent Evaluation

Agent evaluation comes in two phases. 
First you have the "inner loop", which is focused on evaluating the performance of an agent during development.
Then you have the "outer loop", which is focused on evaluating the performance of an agent in a production environment.

<center>
<img src="../1_develop/images/inner_outer_loop.png" alt="drawing" width="75%"/>
<br>
<em>Diagram of the Inner and Outer Loops</em>
</center>

In this section we will focus on the outer loop. The inner loop will be covered in the [Develop section](../1_develop/) of this repo.

In outer loop evaluation, there is no ground truth and the evaluation is done on the live production traces. There are  main stages:

1. Collect and track the agent production traces
2. Build scorers (based on inner loop evaluations) and LLM-judge to evaluate the agent and measure its performance
3. Running the evaluations against the agent traces
4. Setup automatic evaluations in MLflow to continuously monitor the production application
5. Have human experts review the evaluation results and provide their own feedback

We will cover each of these stages in the following sections.

## 1. Generate traces with the NPS Agent deployed on RHOAI

The first step in outer loop evaluation is to make sure we are collecting traces from our production application in MLflow.

You can refer to [2_deploy](../2_deploy) to learn how to deploy your agent application on RHOAI and setup its traces in MLflow.

### Interact with the NPS agent deployed on RHOAI

We will now interact with our deployed NPS agent application (covered in [2_deploy](../2_deploy)) on RHOAI that can answer questions about the U.S. National Park Service (NPS).

This agent has access to an MCP server with five tools:

- `search_parks`: Search for national parks by state, park code, or query
- `get_park_alerts`: Get current alerts for a specific park
- `get_park_campgrounds`: Get campground information for a park
- `get_park_events`: Get upcoming events for a park
- `get_visitor_centers`: Get visitor center information for a park

You can view the code for deploying the NPS agent on RHOAI [here](../2_deploy/). Once the agent is deployed and running on RHOAI it:
- Can be accessed using its exposed URL
- Logs traces in MLflow under the specified experiment name

In [ ]:
# Set the deployed NPS agent URL
AGENT_URL = os.getenv("AGENT_URL")
print(AGENT_URL)

In [ ]:
# Generate traces by querying the deployed NPS agent application
import requests
resp = requests.post(f"{AGENT_URL}/invocations", json={
    "input": [{"role": "user", "content": "What national parks are there in California?"}]
}, timeout=300)

print(resp.json()["output"][0]["content"][0]["text"] if resp.ok else f"Error: {resp.text}")

In [ ]:
resp = requests.post(f"{AGENT_URL}/invocations", json={
    "input": [{"role": "user", "content": "What national parks are there in Oregon?"}]
}, timeout=300)

print(resp.json()["output"][0]["content"][0]["text"] if resp.ok else f"Error: {resp.text}")

In [ ]:
resp = requests.post(f"{AGENT_URL}/invocations", json={
    "input": [{"role": "user", "content": "What campgrounds are available at the Yosemite National Park?"}]
}, timeout=300)

print(resp.json()["output"][0]["content"][0]["text"] if resp.ok else f"Error: {resp.text}")

In [ ]:
resp = requests.post(f"{AGENT_URL}/invocations", json={
    "input": [{"role": "user", "content": "What time does Acadia's main visitor center open during the summer?"}]
}, timeout=300)

print(resp.json()["output"][0]["content"][0]["text"] if resp.ok else f"Error: {resp.text}")

In [ ]:
resp = requests.post(f"{AGENT_URL}/invocations", json={
    "input": [{"role": "user", "content": "What are the current alerts for Yellowstone National Park?"}]
}, timeout=300)

print(resp.json()["output"][0]["content"][0]["text"] if resp.ok else f"Error: {resp.text}")

To view this trace in the MLflow UI, navigate to Experiments -> 'experiment name where the agent application is sending traces' -> Traces. Here you can see all traces that come into your experiment.

Now that we have some traces generated by our deployed agent, we can start evaluating it.

<center>
<img src="./images/mlflow_traces.png" alt="drawing" width="75%"/>
<br>
<em>Diagram of the agent traces</em>
</center>

## 2. Build Scorers and LLM-Judge/Agent-Judge

Next we configure our judges/scorers for our agent.

MLflow comes with a [number of prepackaged LLM-judges](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/predefined/) that can be used for scoring.

We can define custom judges with our own criteria using the `make_judge` function.

In [ ]:
from typing import Literal
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import RelevanceToQuery, ToolCallEfficiency

# Set model
# NOTE: All LLM-judges use LiteLLM, which requires the model name in this format
#       For the OpenAI compatible endpoints, you also need the OPENAI_BASE_URL and OPENAI_API_KEY environment variables set for this to work
model = f'openai:/{os.getenv("OPENAI_MODEL_NAME")}'

nps_evaluator_judge = make_judge(
        name="nps_agent_evaluator",
        instructions=(
            "Evaluate the NPS agent's performance as good, acceptable or poor "
            "by evaluating if the response in\n\n{{ outputs }}\n\n correctly answers the user's question in\n\n{{ inputs }}\n\n"
            "Check for:\n"
            "1. Response Quality: Did the agent correctly answer the user's question "
            "and provide accurate information?\n"
            "2. Tool Usage: Were appropriate and relevant tool calls made in\n\n{{ trace }}\n\n"
            "3. Completeness: Did the agent answer all parts of the "
            "user's question?\n\n"
            "Rate as: 'good', 'acceptable', or 'poor'"
        ),
        feedback_value_type=Literal["good", "acceptable", "poor"],
        model=model,
    )

query_relevancy_judge = RelevanceToQuery(model=model)
tool_call_judge = ToolCallEfficiency(model=model)

scorers = [
        nps_evaluator_judge,
        query_relevancy_judge,                # Checks response is fully relevant to the original query
        tool_call_judge,              # Checks the tool call efficiency
    ]

## 3. Run the Evaluations against the Agent traces

Now we can evaluate the traces we generated using the `evaluate` function.

The evaluate function will take our production traces as the `dataset` object, then score the traces and outputs with the `scorers` we defined.

The results of the evaluation and all the traces will be saved to MLflow.
After running the below script, open MLflow in your browser and navigate to Experiments > Default > Evaluation Runs, then click on the latest run to view the results.

### Fetching the traces
Before running the evaluations, we need to first fetch the traces from the agent application. The `search` function in MLflow allows you to filter your agent traces. You can set the following parameters for fetching the relevant traces:
- `experiment_ids` - The list of experiment ids in MLflow that you want to fetch the traces for
- (optional) `filter_string` - Additional conditions to filter your traces eg: fetch only successful traces, fetch only error traces, fetch traces which have a certain regex pattern etc
- `max_results` - The maximum number of traces to fetch
- `order_by` - Ordering the traces by timestamp

You can find more details in the official MLflow documentation [here](https://mlflow.org/docs/latest/genai/tracing/search-traces/).

For this demo, we will just fetch the latest 5 traces from our NPS agent (that we just generated from step 1)

In [ ]:
# Get the experiment id from the mlflow experiment name for fetching the traces
import mlflow
exp_id = mlflow.get_experiment_by_name(os.getenv("MLFLOW_EXPERIMENT_NAME")).experiment_id
print(exp_id)

In [ ]:
traces = mlflow.search_traces(
        experiment_ids=[exp_id],
        max_results=5,
        order_by=["trace.timestamp_ms DESC"],
    )

In [ ]:
traces

As you can see above, the `search_traces` returns a dataframe with the traces information such as `trace_id`, `state`, `response` etc.

### Running the evaluations

Now that we have fetched the traces, we can run our evaluations against it.

In [ ]:
from mlflow.genai import evaluate

results = evaluate(data=traces, scorers=scorers)

# Print the results
print(results.metrics)

You should see the evaluation results under Evaluation runs in the MLflow UI -> your experiment -> Evaluation runs. 

<center>
<img src="./images/mlflow_eval_runs.png" alt="drawing" width="75%"/>
<br>
<em>Eval run</em>
</center>

You can then click on the evaluation run to see the results of the evaluation scorers.

<center>
<img src="./images/mlflow_eval_trace.png" alt="drawing" width="75%"/>
<br>
<em>Eval trace</em>
</center>

## 4. Human Feedback

Now that you have completed some evaluations, you can share these with your subject matter experts to review and add their own feedback.

If running on a remote tracking server, you can share the link to your evaluation run for them to review and add their feedback.

They can add new feedback or update existing feedback, including the automatic assessments.


### Add New Feedback

<center>
<img src="../1_develop/images/add_feedback.png" alt="drawing" width="75%"/>
<br>
<em>Add feedback</em>
</center>

1. Navigate to your experiment
1. Click "Add Assessment" button
1. Select "Feedback" from the Assessment Type dropdown
1. Enter a descriptive name
1. Choose the appropriate data type
1. Enter your evaluation value
1. Add rationale explaining your evaluation reasoning
1. Click "Create" to record your feedback


### Update Existing Feedback

<center>
<img src="../1_develop/images/edit_feedback.png" alt="drawing" width="75%"/>
<br>
<em>Edit feedback</em>
</center>

1. Locate the feedback entry you want to modify
1. Click the hamburger menu (⋮) next to the feedback entry
1. Select "Edit" from the dropdown menu
1. Modify the value, rationale, or other fields
1. Click "Save" to update the feedback